# Licitaciones Paraguay - DNCP Paraguay

Se utiliza los datos del Portal de Datos Abiertos de la DNCP (Dirección Nacional de Contrataciones Públicas) que contiene información sobre
todas las compras que realiza el Estado paraguayo, clasificadas en etapas:
planificaciones, convocatorias, adjudicaciones, contratos y modificaciones de contrato.

Se descarga el conjunto de datos en formato CSV de los años 2021 y 2022. Un proceso de contratación consta de varias etapas: licitación, adjudicación, contratación e implementación.


Se descarga de estas 2 URLs:
* Adjudicaciones: https://www.contrataciones.gov.py/datos/adjudicaciones  
* Contratos: https://www.contrataciones.gov.py/datos/contratos

Se descargan los archivos de los años 2021 y 2022. Y al descargar se extraen estos archivos de cada año:
* awa_suppliers.csv
* awards.csv
* contracts.csv

Estos archivos están publicados en una carpeta pública de Google Drive y se descargan automáticamente en la siguiente celda con `gdown` (no es necesario montar Drive ni tener acceso privado):
https://drive.google.com/drive/folders/1738grWDl9j2VXf3ju0JFq0Njo2WXiQGo

## Etapa 1 - Carga, Exploración Inicial y Limpieza de Datos

In [ ]:
# Descargar el dataset desde la carpeta publica de Google Drive (no requiere montar Drive)
%pip install -q gdown==5.2.0
import gdown

URL_DATASET = 'https://drive.google.com/drive/folders/1738grWDl9j2VXf3ju0JFq0Njo2WXiQGo'
gdown.download_folder(url=URL_DATASET, output='data', quiet=False)

In [ ]:
# Se importa las primeras librerias para empezar a trabajar con la carga y exploración de los datos
import os
import pandas as pd

In [ ]:
ruta_carpeta = 'data'  # carpeta local donde gdown descargo los CSV

In [ ]:
# Se lista los archivos dentro de la carpeta
listar_archivos = os.listdir(ruta_carpeta)
listar_archivos

In [ ]:
contratos_2021 = pd.read_csv(ruta_carpeta + '/records_contratos_2021.csv', low_memory=False) #archivo de contratos 2021
contratos_2022 = pd.read_csv(ruta_carpeta + '/records_contratos_2022.csv', low_memory=False) #archivo de contratos 2022
contratos = pd.concat([contratos_2021,contratos_2022]) #se unifica en un solo df de contratos

In [ ]:
# Vemos el tamaño del df de contratos
# contratos_2021.shape #9305 registros
# contratos_2022.shape #30652 registros
contratos.shape #39957 registros

In [ ]:
# Creamos el df de adjudicaciones 2021 y 2022
adjudicaciones_2021 = pd.read_csv(ruta_carpeta + '/records_adj_2021.csv', low_memory=False)
adjudicaciones_2022 = pd.read_csv(ruta_carpeta + '/records_adj_2022.csv', low_memory=False)
adjudicaciones = pd.concat([adjudicaciones_2021,adjudicaciones_2022])

In [ ]:
# Vemos el tamaño del df de adjudicaciones
# adjudicaciones_2021.shape #9305 registros
# adjudicaciones_2022.shape #30652 registros
adjudicaciones.shape #39957

Se verifica la misma cantidad de registros de contratos y de adjudicaciones.
Surge la duda de si son los mismos datos duplicados, o solo la misma cantidad con registros diferentes en cada columna.

In [ ]:
contratos.head(3) #

In [ ]:
adjudicaciones.head(3)

Se ven registros identicos. Por lo que se va a unificar y verificar su corresponde a duplicados.

In [ ]:
# Se unifica los datasets
df = pd.concat([contratos,adjudicaciones])
df.shape

In [ ]:
# Se valida duplicados
df.duplicated().sum()

Con esto se confirma que los dataset de records de contratos y adjudicaciones son el mismo, por lo que se procede a avanzar con el dataset de contratos, asignandolo a "df"

In [ ]:
# Se usa contratos como el df a analizar
df = contratos

In [ ]:
df.info()

In [ ]:
df_col = df.columns.tolist()
df_col

In [ ]:
# Renombrar a español - asistido con IA
df_col = [
    'id_contratacion',           # Open Contracting ID
    'id',                        # compiledRelease/id
    'id_licitacion',             # compiledRelease/tender/id
    'titulo',                    # compiledRelease/tender/title
    'estado',                    # compiledRelease/tender/status
    'criterio_adjudicacion',     # compiledRelease/tender/awardCriteria
    'forma_adjudicacion',        # compiledRelease/tender/awardCriteriaDetails
    'forma_presentacion',        # compiledRelease/tender/submissionMethod
    'fecha_apertura_ofertas',    # compiledRelease/tender/bidOpening/date
    'direccion',                 # compiledRelease/tender/bidOpening/address/streetAddress
    'detalles_metodo_envio',     # compiledRelease/tender/submissionMethodDetails
    'criterio_elegibilidad',     # compiledRelease/tender/eligibilityCriteria
    'detalle_estado',            # compiledRelease/tender/statusDetails
    'direccion2',                # compiledRelease/tender/enquiriesAddress/streetAddress
    'categoria_detallada',       # compiledRelease/tender/mainProcurementCategoryDetails
    'hubo_consultas',            # compiledRelease/tender/hasEnquiries
    'monto_licitado',            # compiledRelease/tender/value/amount
    'moneda',                    # compiledRelease/tender/value/currency
    'fecha_publicacion',         # compiledRelease/tender/datePublished
    'fecha_inicio_licitacion',   # compiledRelease/tender/tenderPeriod/startDate
    'fecha_fin_licitacion',      # compiledRelease/tender/tenderPeriod/endDate
    'duracion_dias_licitacion',  # compiledRelease/tender/tenderPeriod/durationInDays
    'fecha_inicio_adjudicacion', # compiledRelease/tender/awardPeriod/startDate
    'fecha_max_extension_contrato', # compiledRelease/tender/contractPeriod/maxExtentDate
    'fecha_fin_consultas',       # compiledRelease/tender/enquiryPeriod/endDate
    'fecha_inicio_consultas',    # compiledRelease/tender/enquiryPeriod/startDate
    'duracion_dias_consultas',   # compiledRelease/tender/enquiryPeriod/durationInDays
    'tipo_bien_servicio',        # compiledRelease/tender/mainProcurementCategory
    'metodo_contratacion',       # compiledRelease/tender/procurementMethod
    'modalidad',                 # compiledRelease/tender/procurementMethodDetails
    'id_convocante',             # compiledRelease/tender/procuringEntity/id
    'convocante',                # compiledRelease/tender/procuringEntity/name
    'idioma',                    # compiledRelease/language
    'ocid',                      # compiledRelease/ocid
    'fecha_compilacion',         # compiledRelease/date
    'tipo_iniciacion',           # compiledRelease/initiationType
    'id_comprador',              # compiledRelease/buyer/id
    'institucion',               # compiledRelease/buyer/name
    'planificacion_id',          # compiledRelease/planning/identifier
    'planificacion_fecha_estimada', # compiledRelease/planning/estimatedDate
    'planificacion_presupuesto_descripcion', # compiledRelease/planning/budget/description
    'planificacion_presupuesto_moneda',      # compiledRelease/planning/budget/amount/currency
    'planificacion_presupuesto_monto',       # compiledRelease/planning/budget/amount/amount
    'etiqueta',                  # compiledRelease/tag
    'contrato_duracion_dias',    # compiledRelease/tender/contractPeriod/durationInDays
    'regimen_especial',          # compiledRelease/tender/coveredBy
    'categorias_adicionales',    # compiledRelease/tender/additionalProcurementCategories
    'num_oferentes',             # compiledRelease/tender/numberOfTenderers  ← TARGET Etapa 3
    'es_subasta_electronica',    # compiledRelease/tender/techniques/hasElectronicAuction
    'intencion_id',              # compiledRelease/tender/procurementIntention/id
    'intencion_uri',             # compiledRelease/tender/procurementIntention/uri
    'intencion_categoria',       # compiledRelease/tender/procurementIntention/category
    'intencion_titulo',          # compiledRelease/tender/procurementIntention/title
    'intencion_descripcion',     # compiledRelease/tender/procurementIntention/description
    'intencion_fecha_inicio',    # compiledRelease/tender/procurementIntention/startDate
    'intencion_fecha_publicacion', # compiledRelease/tender/procurementIntention/publishedDate
    'intencion_entidad_id',      # compiledRelease/tender/procurementIntention/procuringEntity/id
    'intencion_entidad_nombre',  # compiledRelease/tender/procurementIntention/procuringEntity/name
    'intencion_estado',          # compiledRelease/tender/procurementIntention/status
    'intencion_detalle_estado',  # compiledRelease/tender/procurementIntention/statusDetails
    'justificacion_metodo',      # compiledRelease/tender/procurementMethodRationale
    'intencion_justificacion',   # compiledRelease/tender/procurementIntention/rationale
    'segunda_etapa_id',          # compiledRelease/secondStage/id
    'tiene_acuerdo_marco',       # compiledRelease/tender/techniques/hasFrameworkAgreement
    'contrato_fecha_inicio',     # compiledRelease/tender/contractPeriod/startDate
    'contrato_fecha_fin',        # compiledRelease/tender/contractPeriod/endDate
]

In [ ]:
df.columns = df_col
df.columns

In [ ]:
df.info()

In [ ]:
# Se elimina duplicados en caso que haya
df.drop_duplicates()
df.shape

No se observa que haya duplicados.

In [ ]:
pd.set_option('display.max_rows', None) #para ver el listado completo de cuantos valores unicos por columna - Asistido por IA
df.nunique().sort_values()

In [ ]:
pd.reset_option('display.max_rows') #volvemos a la configuracion anterior

In [ ]:
# Se eliminan las columnas con datos de texto libre, o de valores unicos o casi todos nulos ya que aporta para el analisis
cols_eliminar = [
    'idioma',
    'tipo_iniciacion',
    'etiqueta',
    'es_subasta_electronica',
    'intencion_estado',
    'intencion_detalle_estado',
    'intencion_justificacion',
    'tiene_acuerdo_marco',
    'segunda_etapa_id',
    'direccion',
    'direccion2',
    'criterio_elegibilidad',
    'detalles_metodo_envio',
    'planificacion_presupuesto_descripcion',
    'justificacion_metodo',
    'intencion_id',
    'intencion_uri',
    'intencion_categoria',
    'intencion_titulo',
    'intencion_descripcion',
    'intencion_fecha_inicio',
    'intencion_fecha_publicacion',
    'intencion_entidad_id',
    'intencion_entidad_nombre',
    'id_comprador',
    'id_convocante',
    'id_contratacion',
    'id'
]

df = df.drop(columns=cols_eliminar)

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
# Estas columnas también deben ser eliminadas por casi no tener datos
df = df.drop(columns=['contrato_fecha_inicio', 'contrato_fecha_fin'])

In [ ]:
# Se selecciona las columnas de fechas
cols_fechas = [
    'fecha_apertura_ofertas',
    'fecha_publicacion',
    'fecha_inicio_licitacion',
    'fecha_fin_licitacion',
    'fecha_inicio_adjudicacion',
    'fecha_max_extension_contrato',
    'fecha_fin_consultas',
    'fecha_inicio_consultas',
    'fecha_compilacion',
    'planificacion_fecha_estimada',
]

# Se convierte a datetime y usamos coerce para que ponga null en caso de error
for col in cols_fechas:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
# Se selecciona las columnas float para convertir a Int64, para evitar la notación cientifica y para no tener problemas con nulos
cols_float = df.select_dtypes(include='float').columns.tolist()
cols_float

In [ ]:
# Para cada columna float cambiar a Int64
for col in cols_float:
    df[col] = df[col].astype('Int64')

In [ ]:
# Ver un resumen de los datos numéricos
df.describe()

In [ ]:
df.info()

In [ ]:
# Se identifica las columnas que en info se ven con muchos nulos
nulos = df.isna().sum()
nulos[nulos > 25000]

In [ ]:
# Se selecciona las columnas que estan con muchos nulos
cols_muchos_nulos = nulos[nulos > 25000].index.tolist() # Asistido con IA - El index.tolist() estira las columnas que cumplen con la condición
cols_muchos_nulos

In [ ]:
# Se elimina las columnas seleccionadas con muchos nulos del df
df = df.drop(columns=cols_muchos_nulos, errors='ignore')

In [ ]:
# Ver primero los estados de las licitaciones para trabajar con los que nos interesan
print(df['estado'].value_counts())

In [ ]:
# Se filtra solo por los que tienen estado complete para trabajar solo con esta base y entender el comportamiento de las licitaciones completadas.
df = df[df['estado'] == 'complete'].copy()
df.info()

In [ ]:
# Se selecciona las columnas categoricas para entender que datos tiene cada uno agrupado y ver que mas queda por limpiar
cols_categoricas = df.select_dtypes('object')

for col in cols_categoricas:
    print(df[col].value_counts().head(10))
    print()

Hallazgos:

* En ID de licitacion debemos dejar solo el numero para identificar claramente casos unicos.
* En tituo debemos unificar los que son el mismo escritos de diferentes maneras.
* El principal criterio de adjudicacion es es priceonly por lejos.
* La forma de adjudicacion principal es por total.
* La mayoría se presenta en persona.
* Si bien todos están en estado complete, y la mayoría está adjudicada, hay casos donde se cerraron en etapa de evaluacion, otros que quedaron precalificado, otros con convenio vigente o finalizado. Como se decide trabajar con los casos finalizados correctamente, se debe dejar solo los casos adjudicados, filtrando el resto.
* Categoría detallada, podemos trabajar solo con la categoría principal, eliminando lo que hay despues del guión.
* Hubo consultas no parece relevante, podemos filtrar.
* La moneda está principalmente en Gs, pero preferimos trabajar en USD por lo que debemos hacer la conversión.
* El principal metodo de contratación es abierta, luego un hay otros pocos que son directos y marginal son algunos selectivos.
* Entrando mejor en la modalidad, lidera las contrataciones directas, seguido de concurso de oferta, licitación publica y contratación por excepción, estos como top.
* Los principales convocantes son el MSPBS, Municipalidad de Luque, MOPC, y otras Municipalidades.
* OCID no parece relevante, se podría eliminar.
* Institución y convocante parecen ser casi lo mismo, a los efectos de este analisis vamos a ir por institucion, eliminando convocante.
* Planificacion presupuesto moneda, decidimos no avanzar con los datos de planificacion, solo con lo real ya adjudicado.
* También se decide omitir regimen para acotar el análisis a realizar.

In [ ]:
# Se filtra solo por los que tienen detalle_estado ADJUDICADA, ya que como el estado es "complete" los demás parecen ser errores.
df = df[df['detalle_estado'] == 'Adjudicada'].copy()
df.shape

In [ ]:
# El id_licitacion tiene formato largo con texto, eliminamos todo lo que viene despues del guión.
df['id_licitacion'] = df['id_licitacion'].str.split('-').str[0] # str[0] agarra solo el primer elemento, que es el número
print(df['id_licitacion'].head(3))

In [ ]:
# Reemplazamos los titulos similares que probablemente se refieren a lo mismo - Asistido por IA
reemplazos = {
    # Almuerzo escolar
    'PROVISION DE ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'ADQUISICION DE ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'ADQUISICIÓN DE ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'SERVICIO DE ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'SERVICIO DE PROVISION DE ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'ADQUISICION DE ALIMENTOS PARA ALMUERZO ESCOLAR': 'ALMUERZO ESCOLAR',
    'ADQUISICIÓN DEL SERVICIO DE ALMUERZO ESCOLAR - FONACIDE': 'ALMUERZO ESCOLAR',
    'SERVICIO DE ALMUERZO ESCOLAR - FONACIDE': 'ALMUERZO ESCOLAR',
    'ALMUERZO ESCOLAR PARA INSTITUCIONES EDUCATIVAS': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO ESCOLAR, BAJO LA MODALIDAD DE ALIMENTOS PREPARADOS EN LAS ESCUELAS': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO ESCOLAR, BAJO LA MODALIDAD DE ALIMENTOS PREPARADOS EN LAS ESCUELAS (COCINANDO)': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO ESCOLAR, BAJO LA MODALIDAD DE ALIMENTOS PREPARADOS EN LAS INSTITUCIONES EDUCATIVAS (COCINANDO)': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO ESCOLAR, BAJO LA MODALIDAD DE COCINANDO EN LAS INSTITUCIONES EDUCATIVAS': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO/CENA ESCOLAR, BAJO LA MODALIDAD DE ALIMENTOS PREPARADOS EN LAS INSTITUCIONES EDUCATIVAS (COCINANDO)': 'ALMUERZO ESCOLAR',

    # Combustible
    'ADQUISICION DE COMBUSTIBLE': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICIÓN DE COMBUSTIBLE': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICIÓN DE COMBUSTIBLES': 'ADQUISICION DE COMBUSTIBLES',
    'PROVISION DE COMBUSTIBLES': 'ADQUISICION DE COMBUSTIBLES',
    'PROVISIÓN DE COMBUSTIBLES': 'ADQUISICION DE COMBUSTIBLES',
    'COMPRA DE COMBUSTIBLE': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICIÓN DE COMBUSTIBLES PARA USO INSTITUCIONAL': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICION DE COMBUSTIBLES PARA USO INSTITUCIONAL': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICION DE COMBUSTIBLES Y LUBRICANTES': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICIÓN DE COMBUSTIBLE PARA USO INSTITUCIONAL': 'ADQUISICION DE COMBUSTIBLES',
    'ADQUISICIÓN DE COMBUSTIBLES Y LUBRICANTES': 'ADQUISICION DE COMBUSTIBLES',

    # Equipos informaticos
    'ADQUISICIÓN DE EQUIPOS INFORMÁTICOS': 'ADQUISICION DE EQUIPOS INFORMATICOS',
    'ADQUISICIÓN DE EQUIPOS INFORMATICOS': 'ADQUISICION DE EQUIPOS INFORMATICOS',
    'ADQUISICIÓN DE EQUIPOS DE COMPUTACIÓN': 'ADQUISICION DE EQUIPOS INFORMATICOS',
    'ADQUISICION DE EQUIPOS DE COMPUTACION': 'ADQUISICION DE EQUIPOS INFORMATICOS',
    'ADQUISICION DE EQUIPOS DE INFORMATICA': 'ADQUISICION DE EQUIPOS INFORMATICOS',
    'ADQUISICION DE EQUIPOS DE OFICINA Y COMPUTACION': 'ADQUISICION DE EQUIPOS INFORMATICOS',

    # Muebles
    'ADQUISICIÓN DE MUEBLES Y ENSERES': 'ADQUISICION DE MUEBLES Y ENSERES',
    'ADQUISICIÓN DE MUEBLES PARA INSTITUCIONES EDUCATIVAS': 'ADQUISICION DE MUEBLES Y ENSERES',
    'ADQUISICION DE MOBILIARIOS PARA INSTITUCIONES EDUCATIVAS': 'ADQUISICION DE MUEBLES Y ENSERES',
    'ADQUISICIÓN DE MOBILIARIOS PARA INSTITUCIONES EDUCATIVAS': 'ADQUISICION DE MUEBLES Y ENSERES',
    'ADQUISICIÓN DE MUEBLES': 'ADQUISICION DE MUEBLES Y ENSERES',

    # Aulas
    'CONSTRUCCIÓN DE AULAS': 'CONSTRUCCION DE AULA',
    'CONSTRUCCION DE AULAS': 'CONSTRUCCION DE AULA',
    'CONSTRUCCION DE AULA - FONACIDE': 'CONSTRUCCION DE AULA',
    'CONSTRUCCIÓN DE AULA': 'CONSTRUCCION DE AULA',
    'CONSTRUCCION DE AULA ESCOLAR': 'CONSTRUCCION DE AULA',
    'CONSTRUCCION DE UN AULA': 'CONSTRUCCION DE AULA',

    # Empedrado
    'CONSTRUCCIÓN DE PAVIMENTO TIPO EMPEDRADO': 'CONSTRUCCION DE PAVIMENTO TIPO EMPEDRADO',
    'CONSTRUCCIÓN DE EMPEDRADO': 'CONSTRUCCION DE EMPEDRADO',
    'CONSTRUCCIÒN DE EMPEDRADO': 'CONSTRUCCION DE EMPEDRADO',

    # Utiles de oficina
    'ADQUISICIÓN DE ÚTILES DE OFICINA': 'ADQUISICION DE UTILES DE OFICINA',

    # Cubiertas
    'ADQUISICIÓN DE CUBIERTAS': 'ADQUISICION DE CUBIERTAS',

    # Materiales electricos
    'ADQUISICIÓN DE MATERIALES ELÉCTRICOS': 'ADQUISICION DE MATERIALES ELECTRICOS',

    # Elementos de limpieza
    'ADQUISICIÓN DE ELEMENTOS DE LIMPIEZA': 'ADQUISICION DE ELEMENTOS DE LIMPIEZA',

    # Productos alimenticios
    'ADQUISICIÓN DE PRODUCTOS ALIMENTICIOS': 'ADQUISICION DE PRODUCTOS ALIMENTICIOS',

    # Extintores
    'SERVICIO DE RECARGA DE EXTINTORES': 'RECARGA DE EXTINTORES',
}

df['titulo'] = df['titulo'].str.upper().str.strip().replace(reemplazos)
print(df['titulo'].value_counts().head(50))

In [ ]:
reemplazos_adicionales = {
    # Almuerzo escolar
    'PROVISIÓN DE ALMUERZO ESCOLAR EN INSTITUCIONES EDUCATIVAS': 'ALMUERZO ESCOLAR',
    'PROVISION DE ALMUERZO ESCOLAR - FONACIDE': 'ALMUERZO ESCOLAR',
    'PROVISION DE ALMUERZO ESCOLAR EN INSTITUCIONES EDUCATIVAS': 'ALMUERZO ESCOLAR',
    'PROVISIÓN DE ALMUERZO ESCOLAR BAJO LA MODALIDAD DE COCINANDO EN LAS INSTITUCIONES EDUCATIVAS': 'ALMUERZO ESCOLAR',
    'PROVISION DE ALMUERZO ESCOLAR - COMPLEMENTO NUTRICIONAL': 'ALMUERZO ESCOLAR',

    # Equipos de oficina
    'ADQUISICIÓN DE EQUIPOS DE OFICINA': 'ADQUISICION DE EQUIPOS INFORMATICOS',
    'ADQUISICIÓN DE EQUIPOS DE OFICINA Y COMPUTACIÓN': 'ADQUISICION DE EQUIPOS INFORMATICOS',

    # Mobiliario escolar
    'ADQUISICIÓN DE MOBILIARIO ESCOLAR': 'ADQUISICION DE MUEBLES Y ENSERES',
    'ADQUISICIÓN DE MOBILIARIOS ESCOLARES': 'ADQUISICION DE MUEBLES Y ENSERES',

    # Medicamentos
    'ADQUISICIÓN DE MEDICAMENTOS': 'ADQUISICION DE MEDICAMENTOS',

    # Pasajes
    'ADQUISICIÓN DE PASAJES AÉREOS': 'ADQUISICION DE PASAJES AEREOS',

    # Alimentos
    'ADQUISICIÓN DE ALIMENTOS PARA PERSONAS': 'ADQUISICION DE PRODUCTOS ALIMENTICIOS',

    # Utensilios
    'ADQUISICIÓN DE UTENSILIOS DE COCINA Y COMEDOR': 'ADQUISICION DE UTENSILIOS DE COCINA Y COMEDOR',

    # Herramientas
    'ADQUISICIÓN DE HERRAMIENTAS MENORES': 'ADQUISICION DE HERRAMIENTAS MENORES',

    # Textiles
    'ADQUISICIÓN DE TEXTILES Y VESTUARIOS': 'ADQUISICION DE TEXTILES Y VESTUARIOS',

    # Acondicionadores
    'ADQUISICIÓN DE ACONDICIONADORES DE AIRE': 'ADQUISICION DE ACONDICIONADORES DE AIRE',

    # Proyectos
    'SERVICIOS DE ELABORACIÓN DE PROYECTOS DE INVERSIÓN': 'ELABORACION DE PROYECTOS DE INVERSION',
    'ELABORACIÓN DE PROYECTOS DE INVERSIÓN': 'ELABORACION DE PROYECTOS DE INVERSION',

    # Empedrados
    'CONSTRUCCION DE PAVIMENTO TIPO EMPEDRADO': 'CONSTRUCCION DE EMPEDRADO',

    # Aulas
    'CONSTRUCCION DE PAVIMENTO TIPO EMPEDRADO': 'CONSTRUCCION DE EMPEDRADO',
}

reemplazos.update(reemplazos_adicionales)
df['titulo'] = df['titulo'].str.upper().str.strip().replace(reemplazos)
print(df['titulo'].value_counts().head(20))

In [ ]:
# El formato de categoría detallada, limpiamos para dejar solo la categoría
df['categoria_detallada'] = df['categoria_detallada'].str.split(' - ').str[0]
print(df['categoria_detallada'].value_counts())

In [ ]:
# Casi todo está en PYG (guaraníes). Usamos tipo de cambio promedio 2021-2022
tipo_cambio = 7200

# Si la moneda es PYG dividimos por el tipo de cambio, si ya es USD lo dejamos igual
import numpy as np

df['monto_usd'] = np.where( # Asistido con IA np.where(condicion, valor si true, valor si false)
    df['moneda'] == 'USD',
    df['monto_licitado'],
    df['monto_licitado'] / tipo_cambio
).round(2)

df['monto_usd'].head()

In [ ]:
# Eliminamos todo lo que decidimos no usar para acotar el análisis
cols_eliminar = [
    'hubo_consultas',
    'ocid',
    'convocante',
    'planificacion_id',
    'planificacion_fecha_estimada',
    'planificacion_presupuesto_moneda',
    'planificacion_presupuesto_monto',
    'regimen_especial',
    'moneda',
    'monto_licitado',
]

df = df.drop(columns=cols_eliminar, errors='ignore')
df.columns.tolist()

## Etapa 2  -  Análisis Exploratorio de Datos (EDA)

In [ ]:
# Se hace una validación inicial del punto de partida para esta etapa
print(df.shape)
print(df.dtypes)
print(df.head(3))

In [ ]:
# Preparamos todo para armar los graficos - Asistido con IA

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

In [ ]:
# Grafico de distribucion de contrataciones por modalidad y tipo - Asistido por IA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por modalidad
modalidad_counts = df['modalidad'].value_counts().head(6)
axes[0].barh(modalidad_counts.index, modalidad_counts.values, color='steelblue')
axes[0].set_title('Contrataciones por Modalidad')
axes[0].set_xlabel('Cantidad')

# Por tipo de bien/servicio/obra
tipo_counts = df['tipo_bien_servicio'].value_counts()
axes[1].pie(tipo_counts.values, labels=tipo_counts.index, autopct='%1.1f%%', colors=['steelblue','salmon','mediumseagreen'])
axes[1].set_title('Distribución por Tipo')

plt.tight_layout()
plt.show()

* Entre el 2021 y 2022, se puede ver claramente que la mayor cantidad de contrataciones se hacen por modalidad de contratación directa, por mucho, seguido de los concursos de ofertas y licitaciones públicas.
* El 40% son obras, seguido por un 36% de bienes y 24% de servicios

In [ ]:
# Se analiza la mediana por modalidad - Asistido por IA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 6 modalidades por cantidad
top_6_modalidad = df['modalidad'].value_counts().head(6).index

# Monto mediano solo del top 6 por cantidad
monto_modalidad = (
    df[df['modalidad'].isin(top_6_modalidad)]
    .groupby('modalidad')['monto_usd']
    .median()
    .sort_values(ascending=False)
)

axes[0].barh(monto_modalidad.index[::-1], monto_modalidad.values[::-1], color='mediumseagreen')
axes[0].set_title('Monto Mediano por Modalidad (USD) — Top 6 por Cantidad')
axes[0].set_xlabel('Mediana USD')

# Monto por tipo de bien/servicio
monto_tipo = df.groupby('tipo_bien_servicio')['monto_usd'].median().sort_values(ascending=False)
axes[1].bar(monto_tipo.index, monto_tipo.values, color=['steelblue','salmon','mediumseagreen'])
axes[1].set_title('Monto Mediano por Tipo (USD)')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Mediana USD')

plt.tight_layout()
plt.show()

* Acá se puede observar un contraste super interesante, que es que si bien la contratación directa es el top 1 en volumen, son las que salen por importes muy bajos, siendo el mas bajo por contratación en este top 6.
* Y a la inversa, siendo la licitación publica internacional el top 6 en volumen, es la de mayor monto por contratacion, lo que nos dice que de las pocas licitaciones internacionales que salen, son para contratos gigantezcos que suiere podrian ser de mayor impacto.
* Servicios tiene el ticket mediano más alto y es interesante porque contradice lo que se podría esperar, que es que las obras sean mas caras, lo que nos habla de que en obras podriamos tener outliers de importes muy altos.

In [ ]:
# Top de instituciones con mayor cantidad de contratos y por mayor valor monetario - Asistido por IA

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Por cantidad de contratos
top_inst_cant = df['institucion'].value_counts().head(10)
axes[0].barh(top_inst_cant.index[::-1], top_inst_cant.values[::-1], color='steelblue')
axes[0].set_title('Top 10 Instituciones por Cantidad')
axes[0].set_xlabel('Cantidad de contratos')

# Por monto total
top_inst_monto = df.groupby('institucion')['monto_usd'].sum().sort_values(ascending=False).head(10)
axes[1].barh(top_inst_monto.index[::-1], top_inst_monto.values[::-1], color='salmon')
axes[1].set_title('Top 10 Instituciones por Monto (USD)')
axes[1].set_xlabel('Monto total USD')

plt.tight_layout()
plt.show()

* Se puede observar que el MSPBS es la entidad con mayor cantidad de contrataciones y que es la segunda que acumula mayor valor.
* El MOPC es el top 7 en cantidad de contrataciones, pero aun así es el primero con mayor valor, lo que indica que son contratos por valores mucho más altos
* El MDN es top 2 en cantidad pero no aparece en el top de montos, deben ser muchos contratos pequeños, patrón opuesto al MOPC.

In [ ]:
# Para validar el ticket del top 10 instituciones por cantidad, se busca la mediana para entender mejor el volumen por contratación de cada una. - Asistido por IA

top_10_inst = df['institucion'].value_counts().head(10)

ticket_mediano_top10 = (
    df[df['institucion'].isin(top_10_inst.index)]
    .groupby('institucion')['monto_usd']
    .median()
    .round(2)
)

sns.set_theme(style='white') # para excluir las lineas de fondo

# Alinear ambas series por el mismo orden
top_10_ordenado = top_10_inst.sort_values(ascending=False)
ticket_alineado = ticket_mediano_top10.reindex(top_10_ordenado.index)

fig, ax1 = plt.subplots(figsize=(14, 6))

# Eje 1 — barras de cantidad
ax1.bar(top_10_ordenado.index, top_10_ordenado.values, color='steelblue', alpha=0.7, label='Cantidad')
ax1.set_ylabel('Cantidad de Contratos', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_xticklabels(top_10_ordenado.index, rotation=45, ha='right')

# Eje 2 — línea de ticket mediano
ax2 = ax1.twinx()
ax2.plot(top_10_ordenado.index, ticket_alineado.values, color='salmon', marker='o', linewidth=2, label='Ticket Mediano')
ax2.set_ylabel('Ticket Mediano (USD)', color='salmon')
ax2.tick_params(axis='y', labelcolor='salmon')

plt.title('Top 10 Instituciones — Cantidad de Contratos vs Ticket Mediano (USD)')
fig.legend(loc='upper right', bbox_to_anchor=(1, 1), bbox_transform=ax1.transAxes)
plt.tight_layout()
plt.show()

Con esto se puede observar claramente que el MOPC tiene contratos por montos mucho mayores, es un caso atípico, pero que representa que si bien no es la mayor institucion con contrataciones, cada una vale mucho mas que el resto, con una mediana de casi USD 250k, indicando que las inversiones en infraestructura son las de mayor valor, por el cual se puede asumir que también se espera un retorno mejor por el impacto social y economico que debe generar.

## Etapa 3 - Clasificación Supervisada

Se busca saber si la licitación tuvo un unico oferente, que debería ser adjudicación directa, o si fueron multiples oferentes.

In [ ]:
# Se ve la cantidad de licitaciones por cantidad de oferentes.

df['num_oferentes'].value_counts()

La mayoría son de 1 solo oferente, lo que puede indicar la adjudicación directa, pero también hay muchos de multiples oferetens.

In [ ]:
df['num_oferentes'].isna().sum()

Tenemos 1.124 licitaciones donde no tiene cantidad de oferentes, despues vemos como tratarlos.

In [ ]:
# 0 significa sin competencia, uno solo. Y 1 significa con competencia, más de uno

df['competencia'] = (df['num_oferentes'] > 1).astype('Int64') #usamos Int64 para que los valores nulos queden como NA

print(df['competencia'].value_counts())
print(df['competencia'].value_counts(normalize=True).mul(100).round(1)) # Asistido por IA

Se ve que existe un desbalance 65/35, pero no demasiado grande. Hay más registros con competencia.

In [ ]:
print(df.dtypes)
print(f"\nNulos por columna:")
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Se elimina filas donde competencia es nula (no se puede entrenar sin target)
df_model = df.dropna(subset=['competencia']).copy()

# Features seleccionadas
features = [
    'criterio_adjudicacion',
    'titulo',
    'estado',
    'detalle_estado',
    'categoria_detallada',
    'forma_adjudicacion',
    'modalidad',
    'tipo_bien_servicio',
    'metodo_contratacion',
    'monto_usd',
    'duracion_dias_licitacion',
    'institucion'
]

target = 'competencia'

print(f"Nulos en features seleccionadas:")
print(df_model[features].isna().sum())

In [ ]:
# Se elimina los ultimos registros nulos para limpiar el dataset de entrenamiento
df_model = df_model.dropna(subset=['monto_usd', 'duracion_dias_licitacion']).copy()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Definimos la variable para la normalizacion de variables numéricas
scaler = StandardScaler()

# Encoding de variables categóricas
le = LabelEncoder()
cols_categoricas = ['categoria_detallada',
                    'modalidad',
                    'tipo_bien_servicio',
                    'metodo_contratacion',
                    'institucion',
                    'criterio_adjudicacion',
                    'titulo',
                    'estado',
                    'detalle_estado',
                    'forma_adjudicacion']

# Se aplica la normalizacion a las variables categoricas
for col in cols_categoricas:
    df_model[col] = le.fit_transform(df_model[col])

# Se aplica la normalizacion a las variables numericas
df_model['monto_usd'] = scaler.fit_transform(df_model[['monto_usd']])
df_model['duracion_dias_licitacion'] = scaler.fit_transform(df_model[['duracion_dias_licitacion']]) #normalizacion numerica

# Separar features y target
X = df_model[features]
y = df_model[target]

# División train/test 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) #Stratify mantiene la distribucion de las clases en los sets de entranmiento y prueba

print(y_train.value_counts(normalize=True).mul(100).round(1)) # Asistido por IA

Con esto codificamos las variables categoricas y luego en la division nos aseguramos dde que se mantenga la distribucion para evitar sesgos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Se definen los modelos a entrenar para hacer benchmarking
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42), #max_iter en 1000 para tener mucha capacidad de ajustar pesos
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42), #random_state para que los numeros aleatorios sean los mismos para reproducibilidad
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42), #n_estimators es la cantidad de arboles prediciendo
}

# Asistido por IA correr el mismo proceso para cada modelo con menos código
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{nombre} — Accuracy: {acc:.3f}")
    print(classification_report(y_test, y_pred))

In [ ]:
# Asistido por IA
import matplotlib.pyplot as plt

# Recopilamos los resultados obtenidos en el paso anterior
modelos_nombres = ['Regresión Logística', 'Árbol de Decisión', 'Random Forest']
accuracy_scores = [0.652, 0.643, 0.701]

plt.figure(figsize=(10, 6))
plt.bar(modelos_nombres, accuracy_scores, color=['skyblue', 'lightcoral', 'lightgreen'])
plt.ylim(0, 1)  # La precisión va de 0 a 1
plt.title('Comparativa de Accuracy entre Modelos')
plt.ylabel('Accuracy')

# Añadir etiquetas de valor sobre las barras
for i, v in enumerate(accuracy_scores):
    plt.text(i, v + 0.02, str(v), ha='center', fontweight='bold')

plt.show()

* La regresion logistica no es el modelo adecuado para este problema, para los casos sin competencia acertó todos, pero no detectó ningun caso en recall por lo que el F1 también da 0.
* El arbol de desición estuvo un poco mejor, pero le costó identificar los casos sin competencia, como había mas datos con competencia parece que el entrenamiento para esa variable fue mejor.
* Random Forest es el mejor modelo, con un Accuracy del 0.7, pero también mejor presicion y recall para los casos con competencia.

Se selecciona Random Forest como modelo final por su mejor accuracy (0.70) y mejor F1 (0.65), lo que indica que tiene mejor capacidad para generalizar en este caso, que los otros modelos.

## Etapa 4 - Regresión Supervisada





In [ ]:
# Eliminar los nulos de monto_usd del df no normalizado, y los mayores a 0 para asegurar
df_reg = df.dropna(subset=['monto_usd']).copy()
df_reg = df_reg[df_reg['monto_usd'] > 0].copy()

In [ ]:
# Ver outliers con boxplot
df_reg['monto_usd'].plot(kind='box', figsize=(10, 4))
plt.title('Boxplot de monto_usd')
plt.ylabel('USD')
plt.show()

In [ ]:
# Calcular rango intercuartílico (IQR)
Q1 = df_reg['monto_usd'].quantile(0.25)
Q3 = df_reg['monto_usd'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Asistido por IA para los redondeos
print(f"Q1: {Q1:,.0f} USD")
print(f"Q3: {Q3:,.0f} USD")
print(f"IQR: {IQR:,.0f} USD")
print(f"Límite superior: {limite_superior:,.0f} USD")
print(f"Outliers: {((df_reg['monto_usd'] < limite_inferior) | (df_reg['monto_usd'] > limite_superior)).sum()}")

Como en la exploración se vio que habia grupos que en cantidad eran pocos, pero de alto valor, se opta por separar el dataset, y los outlier se van a separar por el limite superior del box plot, para anlizarlos por separado.

In [ ]:
# Partir en dos
df_reg_normal = df_reg[df_reg['monto_usd'] <= limite_superior].copy()
df_reg_grandes = df_reg[df_reg['monto_usd'] > limite_superior].copy()

print(f"Contratos normales: {df_reg_normal.shape[0]:,} registros hasta USD {limite_superior:,.0f}")
print(f"Contratos grandes: {df_reg_grandes.shape[0]:,} registros desde USD {limite_superior:,.0f}")

# Ver distribución de cada uno - Asistido con IA
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_reg_normal['monto_usd'].hist(bins=50, ax=axes[0])
axes[0].set_title('Contratos normales')
axes[0].set_xlabel('USD')

df_reg_grandes['monto_usd'].hist(bins=50, ax=axes[1])
axes[1].set_title('Contratos grandes')
axes[1].set_xlabel('USD')

plt.tight_layout()
plt.show()

Con esta distribucion si veo que no es necesario mantener el de contratos grandes y podemos prescindir y entrenar el modelo con el de contratos normales.

Ahora armamos el modelo de regresión con df_reg_normal:


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Definimos la variable para la normalizacion de variables numéricas
scaler = StandardScaler()

# Features — las mismas que en clasificación
features_reg = ['categoria_detallada',
                'modalidad',
                'tipo_bien_servicio',
                'metodo_contratacion',
                'institucion',
                'criterio_adjudicacion',
                'titulo',
                'estado',
                'detalle_estado',
                'forma_adjudicacion']

# Encoding categóricas
le = LabelEncoder()
for col in features_reg:
    df_reg_normal[col] = le.fit_transform(df_reg_normal[col])

# Se aplica la normalizacion a las variables numericas
df_reg_normal['duracion_dias_licitacion'] = scaler.fit_transform(df_reg_normal[['duracion_dias_licitacion']])

# Eliminar nulos en features
df_reg_normal = df_reg_normal.dropna(subset=features_reg).copy()

# Separar features y target
X_reg = df_reg_normal[features_reg]
y_reg = df_reg_normal['monto_usd']

# División 80/20
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print(f"Train: {X_train_r.shape[0]:,} — Test: {X_test_r.shape[0]:,}")

In [ ]:
# Se definen los modelos y se hace el entrenamiento con la comparación de las métricas
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Se define los modelos para hacer benchmark y definir cual usar
modelos_reg = {
    'Regresión Lineal': LinearRegression(),
    'Árbol de Decisión': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
}

# Asistido por IA correr el mismo proceso para cada modelo con menos código
for nombre, modelo in modelos_reg.items():
    modelo.fit(X_train_r, y_train_r)
    y_pred_r = modelo.predict(X_test_r)
    mae = mean_absolute_error(y_test_r, y_pred_r)
    r2 = r2_score(y_test_r, y_pred_r)
    print(f"\n{nombre}")
    print(f"MAE: {mae:,.0f} USD")
    print(f"R²: {r2:.3f}")

In [ ]:
# Recopilamos los resultados obtenidos en el paso anterior
modelos_nombres_r = ['Regresión Lineal', 'Árbol de Decisión', 'Random Forest']
r2_r = [0.271, 0.441, 0.637]

plt.figure(figsize=(10, 6))
plt.bar(modelos_nombres_r, r2_r, color=['skyblue', 'lightcoral', 'lightgreen'])
plt.ylim(0, 1)  # Va de 0 a 1
plt.title('Comparativa de R2 entre Modelos')
plt.ylabel('R2')

# Añadir etiquetas de valor sobre las barras
for i, v in enumerate(r2_r):
    plt.text(i, v + 0.02, str(v), ha='center', fontweight='bold')

plt.show()

Random Forest es el modelo ganador con R2=0.64 y MAE de USD 8,804. Esto significa que el modelo puede predicir en un 64% el monto de la contratación, con un error promedio de USD 8,804 por contrato.

## Etapa 5 - Clustering

In [ ]:
# Se busca armar el prefil por institucion
df_cluster = df.groupby('institucion').agg( # se agrupa por institución y con .agg usamos para usar diferentes funciones para diferentes variables
    cantidad_contratos = ('monto_usd', 'count'), # se cuenta cuantos contratos tiene cada institucion
    monto_promedio = ('monto_usd', 'mean'), # se calcula el importe promedio
    monto_total = ('monto_usd', 'sum'), # se suma el importe total
    monto_mediano = ('monto_usd', 'median'), # se calcula el importe mediano
    duracion_promedio = ('duracion_dias_licitacion', 'mean'), # se calcula la duracion promedio de dias por licitacion
    modalidad_principal = ('modalidad', lambda x: x.mode()[0]),
    tipo_principal = ('tipo_bien_servicio', lambda x: x.mode()[0]),
).reset_index()

df_cluster = df_cluster.dropna().copy()

print(f"Instituciones: {df_cluster.shape[0]:,}")
print(df_cluster.head())

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Encoding de categóricas
le = LabelEncoder()
df_cluster['modalidad_enc'] = le.fit_transform(df_cluster['modalidad_principal'])
df_cluster['tipo_enc'] = le.fit_transform(df_cluster['tipo_principal'])

# Features para clustering
features_cluster = [
    'cantidad_contratos',
    'monto_promedio',
    'monto_mediano',
    'duracion_promedio',
    'modalidad_enc',
    'tipo_enc',
]

X_cluster = df_cluster[features_cluster].copy()

# Escalar
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Buscar el k óptimo con el método del codo
inercias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inercias.append(km.inertia_)

plt.plot(k_range, inercias, marker='o')
plt.title('Método del Codo — K óptimo')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.xticks(k_range)
plt.show()

Se selecciona clusters

In [ ]:
# Modelo final con k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cluster['cluster'] = km.fit_predict(X_cluster_scaled)

# Ver cuántas instituciones hay en cada cluster
print(df_cluster['cluster'].value_counts().sort_index())

In [ ]:
print("Cluster 2")
print(df_cluster[df_cluster['cluster'] == 2][['institucion', 'cantidad_contratos', 'monto_promedio', 'monto_total']].to_string())

print("\n Cluster 3 ")
print(df_cluster[df_cluster['cluster'] == 3][['institucion', 'cantidad_contratos', 'monto_promedio', 'monto_total']].to_string())

In [ ]:
resumen_clusters = df_cluster.groupby('cluster').agg(
    instituciones = ('institucion', 'count'),
    contratos_promedio = ('cantidad_contratos', 'mean'),
    monto_prom_usd = ('monto_promedio', 'mean'),
    monto_mediano_usd  = ('monto_mediano', 'mean'),
    monto_total_usd = ('monto_total', 'mean'),
    duracion_prom_dias = ('duracion_promedio', 'mean'),
    modalidad_top = ('modalidad_principal', lambda x: x.mode()[0]),
    tipo_top = ('tipo_principal', lambda x: x.mode()[0]),
).round(0)

print(resumen_clusters.to_string())

Con esto podemos perfilar cada cluster

In [ ]:
# Se da nombre a los clusters segun lo que vemos de comportamiento en la tabla anterior
nombres_clusters = {
    0: 'Compradores Medianos de Bienes',
    1: 'Compradores Pequeños de Obras',
    2: 'Mega Contratantes',
    3: 'Grandes Instituciones del Estado',
}

df_cluster['perfil'] = df_cluster['cluster'].map(nombres_clusters)

# Gráfico de burbujas — cantidad vs monto promedio, tamaño = monto total - ASISTIDO CON IA
fig, ax = plt.subplots(figsize=(12, 6))

for cluster, grupo in df_cluster.groupby('cluster'):
    ax.scatter(
        grupo['cantidad_contratos'],
        grupo['monto_promedio'],
        label=nombres_clusters[cluster],
        alpha=0.6,
        s=50
    )

ax.set_title('Segmentación de Instituciones por Perfil de Compra')
ax.set_xlabel('Cantidad de Contratos')
ax.set_ylabel('Monto Promedio (USD)')
ax.legend()
plt.tight_layout()
plt.show()

El clustering KMeans con k=4 identificó 4 perfiles diferentes de instituciones. Los Mega Contratantes que son IPS y MOPC tienen montos promedio mucho mas altos que el resto. Las Grandes Instituciones del Estado tienen un volumen alto y montos variables. Los Compradores Medianos y Pequeños son el resto, la mayoria de las  instituciones (342 de 359) con montos similares y promedio.

## Conclusion

Dataset y limpieza:

Se trabajó con 39.957 registros de contrataciones públicas de la DNCP del 2021 y 2022, que despues de la limpieza quedaron en 23.982 registros con 24 variables. El principal desafío fue el formato con columnas de nombres técnicos.


EDA — Hallazgos principales:

- La Contratación Directa tiene el volumen más alto de contratos pero con los tickets más bajos.
- La Licitación Pública Internacional tiene el ticket mediano más alto, posiblemente de contratos de gran magnitud.
- El MSPBS lidera en cantidad de contratos y el MOPC lidera en valor con un ticket mediano de USD 250k, que hace sentido porque concentra contratos de infraestructura que tienden a ser proyectos grandes.

Clasificación — Etapa 3:

- El mejor modelo fue Random Forest con accuracy 0.70 y F1 macro 0.65.
- El modelo predice bien cuando hay competencia (recall 84%) pero no tanto cuando tiene que predecir sin competencia (recall 44%), tiene sentido por el desbalance del dataset de 65/35.

Regresión — Etapa 4:

- El mejor modelo fue Random Forest con R2=0.64 y MAE de USD 8.804.
- Se trabajó solo con contratos hasta USD 108.099, que era el límite del rango intercuartilico, excluyendo outliers.
- El modelo puede predecir aproximadamente el importe del 64% de las licitaciones, que no es excelente, pero es aceptable.

Clustering — Etapa 5:

Se encontraron 4 perfiles de instituciones:
- Compradores Pequeños de Obras: 228
- Compradores Medianos de Bienes: 114
- Grandes Instituciones del Estado: 15
- Mega Contratantes: 2 (IPS y MOPC)
Estos últimos dos tienen la mayor cantidad de contrataciones del estado paraguayo en estos 2 años.

## Anexo

In [ ]:
# Requerimientos - se detallan las versiones de las librerias
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {seaborn.__version__}")